# **Fase 3: Regularización**

Estudiante: Maria Camila Navarrete Pinzón

Código: 2294353

Fecha: 07 febrero, 2026

# Notebook 7: Regularización

**Objetivo**: Comparar Ridge (L2), Lasso (L1) y ElasticNet

**Conceptos clave**:

**Ridge (L2)**: regParam > 0, elasticNetParam = 0
- Penaliza coeficientes grandes, NO los elimina
**Lasso (L1)**: regParam > 0, elasticNetParam = 1
- Puede eliminar features (coeficientes = 0)
 - Combinación de L1 y L2

**Actividades**:

1. Entrenar modelos con diferentes regularizaciones
2. Comparar resultados
3. Identificar el mejor modelo

## 1. Configuración de SparkSession

Se crea una sesión de spark configurada para ejecutarse en modo local, asignando memoria al driver.

In [8]:

from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col
import pandas as pd

spark = SparkSession.builder \
    .appName("SECOP_Regularizacion") \
    .master("spark://spark-master:7077") \
    .getOrCreate()


## 2. Carga de datos

In [9]:

df = spark.read.parquet("/opt/spark-data/processed/secop_ml_ready.parquet")
df = df.withColumnRenamed("valor_del_contrato_num", "label") \
       .withColumnRenamed("features_pca", "features") \
       .filter(col("label").isNotNull())

train, test = df.randomSplit([0.7, 0.3], seed=42)

print(f"Train: {train.count():,}")
print(f"Test: {test.count():,}")

Train: 36,729


Test: 15,519


## 3. Reto 1: Entender la Regularización

**Pregunta conceptual**: ¿Por qué necesitamos regularización?

**Escenario**: Tu modelo de regresión lineal tiene:
- R² train = 0.95
- R² test = 0.45

El escenario presentado indica que el modelo aprende muy bien los datos de entrenamiento pero falla al generalizar. La regularización introduce una penalización sobre los coeficientes, forzando al modelo a ser más simple y estable, lo que reduce la brecha entre desempeño en train y test.

**Opciones**:
- A) El modelo está underfitting
- B) El modelo está overfitting
- C) El modelo es perfecto
- D) Necesitas más features

**Respuesta: B.)** El modelo está overfitting

**¿Cómo ayuda la regularización en este caso?** La regularización ayuda porque penaliza coeficientes demasiado grandes, reduciendo la complejidad del modelo, evitando que aprenda ruido del conjunto de entrenamiento y mejorando la generalización en datos no vistos.



## 4. Reto 2: Configurar evaluador de modelos

**Objetivo**: Crear un evaluador para comparar modelos.

**Pregunta**: ¿Qué métrica usarías para comparar modelos de regresión?
- RMSE: Penaliza errores grandes
- MAE: Trata todos los errores igual
- R²: Proporción de varianza explicada

Se utiliza **RMSE** porque penaliza con mayor severidad los errores grandes, lo cual es especialmente relevante en este problema donde existen contratos de alto valor. Para comparar modelos de regresión, RMSE permite evaluar no solo precisión promedio sino también estabilidad ante errores extremos.

In [3]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)


## 5. Reto 3: Entrenar modelos con multiples combinaciones L1/L2/ElasticNet

**Objetivo**: Entrenar múltiples modelos variando regParam y elasticNetParam.

**Instrucciones**:
1. Define los valores de `regParam` (lambda) a probar
2. Define los valores de `elasticNetParam` (alpha) a probar
3. Entrena un modelo por cada combinación
4. Registra RMSE de train y test para cada uno

**Parámetros sugeridos**:
- regParam: [0.0, 0.01, 0.1, 1.0, 10.0]
- elasticNetParam: [0.0 (Ridge), 0.5 (ElasticNet), 1.0 (Lasso)]

**Pregunta**: ¿Cuántos modelos entrenarás en total? (combinaciones)

In [4]:
from pyspark.ml.regression import LinearRegression

reg_params = [0.0, 0.01, 0.1, 1.0, 10.0]
elastic_params = [0.0, 0.5, 1.0]  # Ridge, ElasticNet, Lasso

print(f"Combinaciones totales: {len(reg_params) * len(elastic_params)}")

resultados = []

for reg in reg_params:
    for elastic in elastic_params:
        lr = LinearRegression(
            featuresCol="features",
            labelCol="label",
            maxIter=100,
            regParam=reg,
            elasticNetParam=elastic
        )

        model = lr.fit(train)
        predictions = model.transform(test)
        rmse_test = evaluator.evaluate(predictions)

        if reg == 0.0:
            reg_type = "Sin regularización"
        elif elastic == 0.0:
            reg_type = "Ridge (L2)"
        elif elastic == 1.0:
            reg_type = "Lasso (L1)"
        else:
            reg_type = "ElasticNet"

        resultados.append({
            "regParam": reg,
            "elasticNetParam": elastic,
            "tipo": reg_type,
            "rmse_test": rmse_test,
            "rmse_train": model.summary.rootMeanSquaredError,
            "r2_train": model.summary.r2
        })

        print(f"{reg_type:25s} | λ={reg:<5} | α={elastic} | RMSE Test: ${rmse_test:,.2f}")


Combinaciones totales: 15


26/02/13 21:05:17 WARN Instrumentation: [9091cc7f] regParam is zero, which might cause numerical instability and overfitting.
26/02/13 21:05:35 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/02/13 21:05:35 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/02/13 21:05:35 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Sin regularización        | λ=0.0   | α=0.0 | RMSE Test: $5,743,644,041.38


26/02/13 21:05:44 WARN Instrumentation: [84f18b04] regParam is zero, which might cause numerical instability and overfitting.


Sin regularización        | λ=0.0   | α=0.5 | RMSE Test: $5,743,644,041.38


...

Lasso (L1)                | λ=0.1   | α=1.0 | RMSE Test: $5,743,644,041.24


Ridge (L2)                | λ=1.0   | α=0.0 | RMSE Test: $5,743,644,041.28
ElasticNet                | λ=1.0   | α=0.5 | RMSE Test: $5,743,644,040.62
Lasso (L1)                | λ=1.0   | α=1.0 | RMSE Test: $5,743,644,039.95


Ridge (L2)                | λ=10.0  | α=0.0 | RMSE Test: $5,743,644,040.37


ElasticNet                | λ=10.0  | α=0.5 | RMSE Test: $5,743,644,033.74


Lasso (L1)                | λ=10.0  | α=1.0 | RMSE Test: $5,743,644,027.11


A medida que aumenta regParam, el RMSE en test disminuye, lo que indica que el modelo inicial estaba overfitting.
La regularización, especialmente Lasso, mejora la capacidad de generalización al simplificar el modelo.

## 6. Reto 4: Analizar resultados y encontrar mejor modelo

**Objetivo**: Comparar todos los modelos y encontrar el mejor.

**Instrucciones**:

1. Convierte los resultados a un DataFrame de pandas
2. Ordena por RMSE test
3. Identifica el mejor modelo
4. Compara RMSE train vs test para detectar overfitting

**Pregunta**: ¿El mejor modelo es siempre el que tiene menor RMSE en test?

No siempre, aunque el RMSE en test es una métrica fundamental para medir el rendimiento del modelo sobre datos no vistos, no debe ser el único criterio para elegir el mejor modelo. Un RMSE ligeramente menor puede deberse al azar, a una configuración demasiado específica del conjunto de prueba o a un modelo innecesariamente complejo. En la práctica, el mejor modelo es aquel que generaliza bien, no solo el que optimiza una métrica puntual.

**Pregunta**: ¿Qué otros factores considerarías?

- Complejidad del modelo: Modelos con regularización (Ridge, Lasso, ElasticNet) suelen ser más simples y robustos.
Si dos modelos tienen RMSE similares, se elige el más simple (principio de parsimonia).

- Estabilidad del modelo: Un modelo menos sensible a pequeñas variaciones de los datos suele ser más confiable en producción.


In [10]:
import pandas as pd

df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values("rmse_test")
print(df_resultados.to_string(index=False))

mejor_modelo = df_resultados.iloc[0]

print("\nMejor modelo encontrado:")
print(f"Tipo: {mejor_modelo['tipo']}")
print(f"regParam: {mejor_modelo['regParam']}")
print(f"elasticNetParam: {mejor_modelo['elasticNetParam']}")
print(f"RMSE Test: ${mejor_modelo['rmse_test']:,.2f}")


 regParam  elasticNetParam               tipo    rmse_test   rmse_train  r2_train
    10.00              1.0         Lasso (L1) 5.743644e+09 2.398418e+10  0.028727
    10.00              0.5         ElasticNet 5.743644e+09 2.398418e+10  0.028727
     1.00              1.0         Lasso (L1) 5.743644e+09 2.398418e+10  0.028727
    10.00              0.0         Ridge (L2) 5.743644e+09 2.398418e+10  0.028727
     1.00              0.5         ElasticNet 5.743644e+09 2.398418e+10  0.028727
     0.10              1.0         Lasso (L1) 5.743644e+09 2.398418e+10  0.028727
     1.00              0.0         Ridge (L2) 5.743644e+09 2.398418e+10  0.028727
     0.10              0.5         ElasticNet 5.743644e+09 2.398418e+10  0.028727
     0.01              1.0         Lasso (L1) 5.743644e+09 2.398418e+10  0.028727
     0.10              0.0         Ridge (L2) 5.743644e+09 2.398418e+10  0.028727
     0.01              0.5         ElasticNet 5.743644e+09 2.398418e+10  0.028727
     0.01       

El mejor modelo identificado en el experimento fue Lasso (L1) con un valor alto de regularización, lo que sugiere que el modelo original presentaba problemas de overfitting y que era necesario controlar su complejidad para mejorar la generalización. La regularización L1 resultó especialmente efectiva porque no solo reduce la magnitud de los coeficientes, sino que también elimina variables poco relevantes, lo cual es útil en escenarios donde existen features redundantes o con bajo aporte predictivo. El hecho de que un regParam elevado produzca el menor RMSE en el conjunto de prueba indica que priorizar un modelo más simple permitió obtener un mejor desempeño en datos no vistos, logrando un equilibrio adecuado entre sesgo y varianza.

## 7. Reto 5: Comparar overfitting entre tipos de regularizacion

**Objetivo**: Analizar la brecha entre train y test para cada tipo de regularización.

**Instrucciones**:
1. Calcula la diferencia RMSE_test - RMSE_train para cada modelo
2. ¿Qué tipo de regularización reduce más el overfitting?
3. ¿Hay un trade-off entre overfitting y rendimiento general?

**Pregunta de análisis**:
- Si regParam=0.0 tiene RMSE_train muy bajo pero RMSE_test alto → ¿overfitting?

Esto significa que el modelo se ajusta demasiado bien a los datos de entrenamiento, aprendiendo incluso el ruido, pero no logra generalizar a datos nuevos. La ausencia de regularización permite coeficientes grandes y un modelo excesivamente complejo.

- Si regParam=10.0 tiene RMSE_train y RMSE_test ambos altos → ¿underfitting

En este caso, la regularización es tan fuerte que restringe demasiado los coeficientes, impidiendo que el modelo capture los patrones reales de los datos. El resultado es un modelo demasiado simple que se desempeña mal tanto en entrenamiento como en prueba.?


In [11]:
for _, row in df_resultados.iterrows():
    gap = row['rmse_test'] - row['rmse_train']
    print(f"{row['tipo']:25s} | Gap RMSE: ${gap:,.2f}")


Lasso (L1)                | Gap RMSE: $-18,240,533,329.00
ElasticNet                | Gap RMSE: $-18,240,533,322.37
Lasso (L1)                | Gap RMSE: $-18,240,533,316.15
Ridge (L2)                | Gap RMSE: $-18,240,533,315.73
ElasticNet                | Gap RMSE: $-18,240,533,315.49
Lasso (L1)                | Gap RMSE: $-18,240,533,314.87
Ridge (L2)                | Gap RMSE: $-18,240,533,314.83
ElasticNet                | Gap RMSE: $-18,240,533,314.80
Lasso (L1)                | Gap RMSE: $-18,240,533,314.74
Ridge (L2)                | Gap RMSE: $-18,240,533,314.74
ElasticNet                | Gap RMSE: $-18,240,533,314.73
Ridge (L2)                | Gap RMSE: $-18,240,533,314.73
Sin regularización        | Gap RMSE: $-18,240,533,314.73
Sin regularización        | Gap RMSE: $-18,240,533,314.73
Sin regularización        | Gap RMSE: $-18,240,533,314.73


Los modelos con regularización Ridge y ElasticNet tienden a reducir más el overfitting al disminuir la brecha entre error de entrenamiento y prueba. Valores muy altos de regParam generan underfitting, ya que el modelo se vuelve demasiado rígido.

## 8. Reto 6: Entrenar y guardar modelo final

**Objetivo**: Entrenar el modelo con los mejores hiperparámetros.

**Instrucciones**:
1. Usa los hiperparámetros del mejor modelo encontrado
2. Entrena con todos los datos de train
3. Evalúa en test
4. Guarda el modelo

In [12]:
lr_final = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=mejor_modelo['regParam'],
    elasticNetParam=mejor_modelo['elasticNetParam']
)

modelo_final = lr_final.fit(train)


In [13]:
model_path = "/opt/spark-data/processed/regularized_model"
modelo_final.write().overwrite().save(model_path)
print(f"Modelo final guardado en: {model_path}")


Modelo final guardado en: /opt/spark-data/processed/regularized_model


## 9. Bonus 1: Visualizar efecto de lambda en coeficientes Lasso

**Objetivo**: Visualizar cómo lambda afecta los coeficientes del modelo.

**Instrucciones**:
1. Para cada valor de regParam, entrena un modelo Lasso (L1)
2. Extrae los coeficientes
3. Cuenta cuántos coeficientes son exactamente 0
4. ¿A mayor lambda, más coeficientes eliminados?

**Pregunta**: ¿Por qué Lasso puede poner coeficientes en 0 pero Ridge no? 

Lasso puede llevar coeficientes exactamente a cero porque usa penalización L1, que favorece soluciones dispersas. Ridge solo reduce magnitudes, pero no elimina variables completamente.

In [14]:
import numpy as np

for reg in [0.01, 0.1, 1.0, 10.0]:
    lr_lasso = LinearRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=100,
        regParam=reg,
        elasticNetParam=1.0
    )

    model = lr_lasso.fit(train)
    coefs = np.array(model.coefficients)
    zeros = np.sum(np.abs(coefs) < 1e-6)

    print(f"λ={reg:<5} | Coeficientes en 0: {zeros}/{len(coefs)}")


λ=0.01  | Coeficientes en 0: 0/10


λ=0.1   | Coeficientes en 0: 0/10


λ=1.0   | Coeficientes en 0: 0/10


λ=10.0  | Coeficientes en 0: 0/10


## 10. Preguntas de Reflexión

**¿Cuándo usarías Ridge vs Lasso vs ElasticNet?**

*Respuesta:*

- Ridge: muchas features correlacionadas
- Lasso: selección de variables
- ElasticNet: combinación de ambas

**¿Qué pasa si regParam es demasiado grande?**

*Respuesta:* El modelo entra en underfitting y pierde capacidad predictiva.

**¿Es posible que el modelo sin regularización sea el mejor?**

*Respuesta:* Si el dataset es limpio, pequeño y bien balanceado, si.

**¿Cómo elegirías el valor óptimo de regParam en producción?**

*Respuesta:* Con validación cruzada y monitoreo continuo del desempeño en datos reales.

In [15]:
import json

with open("/opt/spark-data/processed/regularizacion_resultados.json", "w") as f:
    json.dump(resultados, f, indent=2)

print("Resultados de regularización guardados")


Resultados de regularización guardados


In [17]:
print("Resumen regularización")
print("Verifica que hayas completado:")
print("  [✓] Entendido diferencia entre L1, L2 y ElasticNet")
print("  [✓] Experimentado con múltiples combinaciones")
print("  [✓] Identificado el mejor modelo")
print("  [✓] Analizado overfitting vs underfitting")
print("  [✓] Guardado modelo final")


Resumen regularización
Verifica que hayas completado:
  [✓] Entendido diferencia entre L1, L2 y ElasticNet
  [✓] Experimentado con múltiples combinaciones
  [✓] Identificado el mejor modelo
  [✓] Analizado overfitting vs underfitting
  [✓] Guardado modelo final


In [18]:
spark.stop()